# Generation G - S1E3 - Cache

This notebook is the companion of posts about Generative AI.

This episode shows how to use the OpenAI lmm with cache.

## Conclusion 

other types of caching (sqllite)

seconds to milli-seconds

exatc match not sremaintic

# Material

## Initializations

In [2]:
### Update environment

In [3]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

Hit:1 http://deb.debian.org/debian bullseye InRelease
Hit:2 http://security.debian.org/debian-security bullseye-security InRelease
Hit:3 http://deb.debian.org/debian bullseye-updates InRelease
Reading package lists... Done


In [4]:
!apt-get update && apt-get install -y jq 1>/dev/null

Hit:1 http://security.debian.org/debian-security bullseye-security InRelease
Hit:2 http://deb.debian.org/debian bullseye InRelease
Hit:3 http://deb.debian.org/debian bullseye-updates InRelease
Reading package lists... Done


In [5]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [6]:
!pip install langchain==0.0.230 1>/dev/null

In [7]:
!pip install openai==0.27.8 1>/dev/null

In [8]:
!pip install tiktoken==0.4.0 1>/dev/null

## Secrets and credentials

In [9]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [10]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


## Code session

LangChain provides an optional caching layer for LLMs. This is useful for two reasons:

It can save you money by reducing the number of API calls you make to the LLM provider, if you're often requesting the same completion multiple times. It can speed up your application by reducing the number of API calls you make to the LLM provider.


https://python.langchain.com/docs/modules/model_io/models/llms/llm_caching

In [12]:
import langchain
from langchain.llms import OpenAI

# To make the caching really obvious, lets use a slower model.
llm = OpenAI(model_name="text-davinci-002", n=2, best_of=2)

In [13]:
%%time
from langchain.cache import InMemoryCache
# no cache

# The first time, it is not yet in cache, so it should take longer
llm.predict("Tell me a joke")

CPU times: user 15.8 ms, sys: 3.48 ms, total: 19.3 ms
Wall time: 543 ms


'\n\nWhy did the chicken cross the road?\n\nTo get to the other side.'

In [14]:
%%time
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

# The first time, it is not yet in cache, so it should take longer
llm.predict("Tell me a joke")

CPU times: user 2.69 ms, sys: 3.64 ms, total: 6.33 ms
Wall time: 713 ms


'\n\nWhy did the chicken cross the road?\n\nTo get to the other side.'

In [15]:
%%time
# The first time, it is not yet in cache, so it should take longer
llm.predict("Tell me a joke")

CPU times: user 215 µs, sys: 25 µs, total: 240 µs
Wall time: 244 µs


'\n\nWhy did the chicken cross the road?\n\nTo get to the other side.'

In [16]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



The White Rabbit is a character from the book Alice's Adventures in Wonderland by Lewis Carroll. The White Rabbit is a white rabbit who is always late.
CPU times: user 5.19 ms, sys: 2.13 ms, total: 7.32 ms
Wall time: 799 ms


In [17]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



The White Rabbit is a character from the book Alice's Adventures in Wonderland by Lewis Carroll. The White Rabbit is a white rabbit who is always late.
CPU times: user 351 µs, sys: 0 ns, total: 351 µs
Wall time: 617 µs


In [18]:
# cache reset
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

In [19]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



In Alice's Adventures in Wonderland, the White Rabbit is a fictional character who appears at the very beginning of the book, in chapter one. He is always in a hurry and is late for important appointments.
CPU times: user 16.9 ms, sys: 1.26 ms, total: 18.1 ms
Wall time: 931 ms


In [20]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



In Alice's Adventures in Wonderland, the White Rabbit is a fictional character who appears at the very beginning of the book, in chapter one. He is always in a hurry and is late for important appointments.
CPU times: user 303 µs, sys: 37 µs, total: 340 µs
Wall time: 631 µs


In [ ]:
# flexibility

In [21]:
%%time
query = "Tell me who is the White Rabbit?"
print(llm(query))



The White Rabbit is a fictional character in Lewis Carroll's 1865 novel Alice's Adventures in Wonderland. He appears at the very beginning of the book, in chapter one, wearing a waistcoat, and muttering "Oh dear! Oh dear! I shall be too late!" Alice follows him down a rabbit hole into Wonderland.
CPU times: user 0 ns, sys: 6.37 ms, total: 6.37 ms
Wall time: 941 ms


In [22]:
%%time
query = "Tell me who is the White Rabbit?"
print(llm(query))



The White Rabbit is a fictional character in Lewis Carroll's 1865 novel Alice's Adventures in Wonderland. He appears at the very beginning of the book, in chapter one, wearing a waistcoat, and muttering "Oh dear! Oh dear! I shall be too late!" Alice follows him down a rabbit hole into Wonderland.
CPU times: user 263 µs, sys: 33 µs, total: 296 µs
Wall time: 303 µs


In [23]:
# cancel cache
langchain.llm_cache = None

In [24]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



The White Rabbit is a character from the children's book Alice's Adventures in Wonderland by Lewis Carroll. The White Rabbit is late for a very important date and ends up leading Alice down a rabbit hole into a fantastical world.
CPU times: user 19.5 ms, sys: 246 µs, total: 19.7 ms
Wall time: 812 ms


In [25]:
%%time
query = "Who is the White Rabbit?"
print(llm(query))



The White Rabbit is a character in the novel Alice's Adventures in Wonderland by Lewis Carroll.
CPU times: user 6.48 ms, sys: 1.36 ms, total: 7.84 ms
Wall time: 874 ms


In [29]:
%%time
from langchain.callbacks import get_openai_callback
from langchain.cache import InMemoryCache

# cancel cache
langchain.llm_cache = None

with get_openai_callback() as cb:
    langchain.llm_cache = InMemoryCache()
    for _ in range(3):
        query = "What is the distance to the Moon?"
        response = llm(query)
        print(response)
        
        query = "Who is the White Rabbit?"
        response = llm(query)
        print(response)
    
        print("\n")
        print(cb)
        nr_tokens_used = cb.total_tokens 
        total_cost = cb.total_cost



The distance to the Moon is 3.844 thousand kilometers.


In Alice in Wonderland, the White Rabbit is a character who is always late and in a hurry. He represents the chaotic and unpredictable nature of life.


Tokens Used: 106
	Prompt Tokens: 14
	Completion Tokens: 92
Successful Requests: 2
Total Cost (USD): $0.0021200000000000004


The distance to the Moon is 3.844 thousand kilometers.


In Alice in Wonderland, the White Rabbit is a character who is always late and in a hurry. He represents the chaotic and unpredictable nature of life.


Tokens Used: 106
	Prompt Tokens: 14
	Completion Tokens: 92
Successful Requests: 2
Total Cost (USD): $0.0021200000000000004


The distance to the Moon is 3.844 thousand kilometers.


In Alice in Wonderland, the White Rabbit is a character who is always late and in a hurry. He represents the chaotic and unpredictable nature of life.


Tokens Used: 106
	Prompt Tokens: 14
	Completion Tokens: 92
Successful Requests: 2
Total Cost (USD): $0.002120000000000